In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

INTERIM_PATH = Path('../data/interim/merged_inspections_licenses_inner.csv')
PROCESSED_PATH = Path('../data/processed/merged_inspections_licenses_inner_clean.csv')
PROCESSED_PARQUET_PATH = Path('../data/processed/merged_inspections_licenses_inner_clean.parquet')
LOG_PATH = Path('../data/interim/cleaning_log.csv')
QUARANTINE_PATH = Path('../data/interim/quarantine.csv')

df_raw = pd.read_csv(INTERIM_PATH)
df_clean = df_raw.copy()

print('Loaded raw shape:', df_raw.shape)

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\interim\\merged_inspections_licenses_inner.csv'

In [ ]:
def profile(df: pd.DataFrame, label: str) -> pd.DataFrame:
    missing = df.isna().sum().sort_values(ascending=False)
    missing_pct = (missing / len(df) * 100).round(2)

    out = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': missing_pct,
        'dtype': df.dtypes.astype(str)
    }).sort_values(['missing_count', 'dtype'], ascending=[False, True])

    print(f'=== {label} ===')
    print('shape:', df.shape)
    print('full duplicates:', int(df.duplicated().sum()))
    print('duplicate Inspection ID:', int(df.duplicated(subset=['Inspection ID']).sum()))
    return out

log_rows = []

def log_step(step: str, rows_before: int, rows_after: int, cols_before: int, cols_after: int, note: str = '') -> None:
    log_rows.append({
        'step': step,
        'rows_before': rows_before,
        'rows_after': rows_after,
        'rows_removed': rows_before - rows_after,
        'cols_before': cols_before,
        'cols_after': cols_after,
        'cols_removed': cols_before - cols_after,
        'note': note,
    })

pre_profile = profile(df_clean, 'Pre-clean profile')
pre_profile.head(20)

=== Pre-clean profile ===
shape: (196825, 59)
full duplicates: 149
duplicate Inspection ID: 149


,missing_count,missing_pct,dtype
Census Tracts,196825,100.00,float64
Community Areas,196825,100.00,float64
Historical Wards 2003-2015,196825,100.00,float64
Wards,196825,100.00,float64
Zip Codes,196825,100.00,float64
APPLICATION CREATED DATE,178239,90.56,object
LICENSE STATUS CHANGE DATE,155935,79.23,object
SSA,134988,68.58,float64
Violations,52266,26.55,object
BUSINESS ACTIVITY,20600,10.47,object


In [ ]:
# Explore 1: Identify fully-null columns
all_null_cols = [c for c in df_clean.columns if df_clean[c].isna().all()]
print('All-null columns:', all_null_cols)

All-null columns: ['Historical Wards 2003-2015', 'Zip Codes', 'Community Areas', 'Census Tracts', 'Wards']


In [ ]:
# Action 1: Drop fully-null columns
rows_before, cols_before = df_clean.shape
df_clean = df_clean.drop(columns=all_null_cols)
log_step('drop_all_null_columns', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='; '.join(all_null_cols))
print('Shape after dropping fully-null columns:', df_clean.shape)

Shape after dropping fully-null columns: (196825, 54)


In [ ]:
# Explore 2: Duplicate behavior
dup_rows = df_clean[df_clean.duplicated(subset=['Inspection ID'], keep=False)].sort_values('Inspection ID')
print('Rows in duplicate Inspection ID groups:', len(dup_rows))
print('Fully duplicated rows:', int(df_clean.duplicated().sum()))

if len(dup_rows) > 0:
    sample_id = dup_rows['Inspection ID'].iloc[0]
    sample_dup = dup_rows[dup_rows['Inspection ID'] == sample_id]
    print('Sample duplicate Inspection ID:', sample_id)
    print('Rows in sample group:', len(sample_dup))
    print('Sample rows identical?', sample_dup.nunique(dropna=False).eq(1).all())

Rows in duplicate Inspection ID groups: 298
Fully duplicated rows: 149
Sample duplicate Inspection ID: 1386194
Rows in sample group: 2
Sample rows identical? True


In [ ]:
print(df_clean.columns)

Index(['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'City', 'State', 'Zip', 'Inspection Date', 'Inspection Type', 'Results',
       'Violations', 'Latitude', 'Longitude', 'Location', 'BL_ID', 'BL_LICENSE_ID', 'ACCOUNT NUMBER', 'SITE NUMBER', 'BL_LEGAL_NAME', 'BL_DBA_NAME', 'BL_ADDRESS', 'BL_CITY',
       'BL_STATE', 'BL_ZIP_CODE', 'WARD', 'PRECINCT', 'WARD PRECINCT', 'POLICE DISTRICT', 'COMMUNITY AREA', 'COMMUNITY AREA NAME', 'NEIGHBORHOOD', 'LICENSE CODE',
       'LICENSE DESCRIPTION', 'BUSINESS ACTIVITY ID', 'BUSINESS ACTIVITY', 'LICENSE NUMBER', 'APPLICATION TYPE', 'APPLICATION CREATED DATE', 'APPLICATION REQUIREMENTS COMPLETE',
       'PAYMENT DATE', 'CONDITIONAL APPROVAL', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE', 'LICENSE APPROVED FOR ISSUANCE', 'DATE ISSUED', 'LICENSE STATUS',
       'LICENSE STATUS CHANGE DATE', 'SSA', 'BL_LATITUDE', 'BL_LONGITUDE', 'BL_LOCATION'],
      dtype='object')


In [ ]:
# Action 2: Remove duplicates (exact first, then by Inspection ID)
rows_before, cols_before = df_clean.shape

df_clean = df_clean.drop_duplicates(keep='first')
log_step('drop_exact_duplicates', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Drop fully duplicated rows')

rows_before, cols_before = df_clean.shape

# This part has no major significance given the sample duplicate rows are identical, 
# but in case there are multiple inspections with the same ID but different dates, 
# we want to keep the latest one. 
# If 'Inspection Date' is not present, it will just keep the first occurrence of each 'Inspection ID'.

if 'Inspection Date' in df_clean.columns:
    sort_dates = pd.to_datetime(df_clean['Inspection Date'], errors='coerce')
    df_clean = df_clean.assign(_sort_inspection_date=sort_dates)
    df_clean = df_clean.sort_values(['Inspection ID', '_sort_inspection_date'], ascending=[True, False]).drop(columns=['_sort_inspection_date'])

df_clean = df_clean.drop_duplicates(subset=['Inspection ID'], keep='first')
log_step('drop_duplicate_inspection_id', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Keep latest Inspection Date per Inspection ID')

# Hard guard against row explosion and duplicate IDs after dedup.
expected_rows_after_dedup = 196_825
tolerance = int(expected_rows_after_dedup * 0.05)
actual_rows_after_dedup = df_clean.shape[0]
print('Shape after duplicate handling:', df_clean.shape)
print(f'Rows after dedup: {actual_rows_after_dedup:,} (expected ~{expected_rows_after_dedup:,} +/- {tolerance:,})')
assert abs(actual_rows_after_dedup - expected_rows_after_dedup) <= tolerance, 'Post-dedup row count is unexpectedly far from expected baseline.'
assert int(df_clean.duplicated(subset=['Inspection ID']).sum()) == 0, 'Duplicate Inspection ID values still remain after dedup.'

Shape after duplicate handling: (196676, 55)
Rows after dedup: 196,676 (expected ~196,825 +/- 9,841)


In [ ]:
# Explore 3: City and State anomalies before normalization
city_values = df_clean['City'].dropna().astype(str)
state_values = df_clean['State'].dropna().astype(str)

print('Unique CITY values after upper-strip:', city_values.str.upper().str.strip().nunique())
print('Top 20 CITY values (raw):')
print(city_values.value_counts().head(20))

unexpected_states = sorted(set(state_values.str.upper().str.strip()) - {'IL'})
print('\nUnexpected STATE values:', unexpected_states)
print('Count rows with State != IL after normalization logic:', int((state_values.str.upper().str.strip() != 'IL').sum()))

KeyError: 'City'

In [ ]:
# Action 3: Normalize key text fields, add city/state flags, then drop near-constant columns
rows_before, cols_before = df_clean.shape
for col in ['City', 'State', 'DBA Name', 'AKA Name', 'Address']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('string').str.strip().str.upper()

if 'City' in df_clean.columns:
    city_norm = df_clean['City'].str.upper().str.replace(r'\s+', ' ', regex=True)
    city_norm = city_norm.str.replace(r'^C+HICAGO$', 'CHICAGO', regex=True)
    df_clean['City'] = city_norm

if 'State' in df_clean.columns:
    df_clean['State'] = df_clean['State'].str.upper()

if 'State' in df_clean.columns:
    df_clean['flag_non_il_state'] = df_clean['State'].notna() & (df_clean['State'] != 'IL')
if 'City' in df_clean.columns:
    df_clean['flag_non_chicago_city'] = df_clean['City'].notna() & (df_clean['City'] != 'CHICAGO')

drop_geo_cols = [c for c in ['City', 'State'] if c in df_clean.columns]
if drop_geo_cols:
    df_clean = df_clean.drop(columns=drop_geo_cols)

log_step('normalize_text_fields_and_flags', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Normalize text, add flags, drop City/State')
print('Shape after city/state normalization:', df_clean.shape)

Shape after city/state normalization: (196676, 55)


In [ ]:
# Explore 4: Longitude distribution and out-of-range candidates
longitude = pd.to_numeric(df_clean['Longitude'], errors='coerce')
print('Longitude quantiles:')
print(longitude.quantile([0.0, 0.01, 0.05, 0.5, 0.95, 0.99, 1.0]))
print('Count < -87.95:', int((longitude < -87.95).sum()))
print('Count > -87.5:', int((longitude > -87.5).sum()))
print('Sample low longitude values:', longitude[longitude < -87.95].dropna().head(10).tolist())

Longitude quantiles:
0.00   -87.914428
0.01   -87.914428
0.05   -87.778516
0.50   -87.666841
0.95   -87.601401
0.99   -87.552631
1.00   -87.525094
Name: Longitude, dtype: float64
Count < -87.95: 0
Count > -87.5: 0
Sample low longitude values: []


In [ ]:
# From the exploration in the cell above, there's no anomalies
# For reproducibility, we'll still add the flagging logic in case there are out-of-range values in other datasets or future updates.
# Action 4: Add longitude quality flag (preserve rows)
rows_before, cols_before = df_clean.shape

if 'Longitude' in df_clean.columns:
    df_clean['flag_longitude_outside_typical_range'] = (
        df_clean['Longitude'].notna() & ((df_clean['Longitude'] < -87.95) | (df_clean['Longitude'] > -87.5))
    )

log_step('add_longitude_flag', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Flag out-of-range longitude values')
print('Flagged longitude anomalies:', int(df_clean['flag_longitude_outside_typical_range'].sum()))

Flagged longitude anomalies: 0


In [ ]:
# Explore 5: ZIP and dtypes before casting/normalization
zip_num = pd.to_numeric(df_clean['Zip'], errors='coerce')
print('ZIP min/max (numeric):', zip_num.min(), zip_num.max())
print('ZIP null count:', int(zip_num.isna().sum()))
print('Sample raw ZIP values:', df_clean['Zip'].dropna().head(10).tolist())

print('\nCurrent dtypes for selected columns:')
inspect_cols = ['Inspection Date', 'APPLICATION CREATED DATE', 'PAYMENT DATE', 'Risk', 'Zip', 'License #', 'LICENSE NUMBER']
for c in inspect_cols:
    if c in df_clean.columns:
        print(c, '->', df_clean[c].dtype)

ZIP min/max (numeric): 10014.0 60827.0
ZIP null count: 50
Sample raw ZIP values: [60657.0, 60657.0, 60657.0, 60657.0, 60657.0, 60614.0, 60657.0, 60640.0, 60657.0, 60657.0]

Current dtypes for selected columns:
Inspection Date -> object
APPLICATION CREATED DATE -> object
PAYMENT DATE -> object
Risk -> object
Zip -> float64
License # -> float64
LICENSE NUMBER -> float64


In [ ]:
print(df_clean['Location'].head(5))

196810    {'latitude': '-87.64910277148812', 'longitude'...
195961    {'latitude': '-87.66862496860755', 'longitude'...
195986    {'latitude': '-87.65381102535457', 'longitude'...
195059    {'latitude': '-87.63680583708285', 'longitude'...
195096    {'latitude': '-87.63680583708285', 'longitude'...
Name: Location, dtype: object


In [ ]:
# Action 5: Cast types, remove leakage, build informative flags, and clean feature columns
rows_before, cols_before = df_clean.shape

int_like_cols = [
    'Inspection ID', 'License #', 'LICENSE NUMBER', 'BL_LICENSE_ID', 'ACCOUNT NUMBER',
    'SITE NUMBER', 'WARD', 'PRECINCT', 'POLICE DISTRICT', 'COMMUNITY AREA',
    'LICENSE CODE', 'SSA', 'BL_ZIP_CODE'
]

for col in int_like_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').round().astype('Int64')

float_cols = ['Latitude', 'Longitude', 'BL_LATITUDE', 'BL_LONGITUDE']
for col in float_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

date_cols = [
    'Inspection Date', 'APPLICATION CREATED DATE', 'APPLICATION REQUIREMENTS COMPLETE',
    'PAYMENT DATE', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE',
    'LICENSE APPROVED FOR ISSUANCE', 'DATE ISSUED', 'LICENSE STATUS CHANGE DATE'
]

for col in date_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

if 'Zip' in df_clean.columns:
    zip_num = pd.to_numeric(df_clean['Zip'], errors='coerce')
    df_clean['Zip'] = zip_num.astype('Int64').astype('string').str.zfill(5)
    df_clean.loc[zip_num.isna(), 'Zip'] = pd.NA

if 'Risk' in df_clean.columns:
    risk_raw = df_clean['Risk'].astype('string').str.strip()
    risk_map = {
        'Risk 1 (High)': 'High',
        'Risk 2 (Medium)': 'Medium',
        'Risk 3 (Low)': 'Low',
        'All': pd.NA
    }
    risk_clean = risk_raw.replace(risk_map)

    if 'Facility Type' in df_clean.columns:
        risk_mode_by_facility = risk_clean.groupby(df_clean['Facility Type']).transform(
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else pd.NA
        )
        risk_clean = risk_clean.fillna(risk_mode_by_facility)

    df_clean['Risk'] = risk_clean.fillna('Unknown')

if {'Inspection Date', 'LICENSE TERM START DATE'}.issubset(df_clean.columns):
    future_license_start_mask = (
        df_clean['Inspection Date'].notna()
        & df_clean['LICENSE TERM START DATE'].notna()
        & (df_clean['LICENSE TERM START DATE'] > df_clean['Inspection Date'])
    )
    df_clean.loc[future_license_start_mask, 'LICENSE TERM START DATE'] = pd.NaT
else:
    future_license_start_mask = pd.Series(False, index=df_clean.index)

if 'Violations' in df_clean.columns:
    violations_str = df_clean['Violations'].astype('string').str.strip()
    df_clean['violations_recorded'] = violations_str.notna() & violations_str.ne('')

if 'BL_LICENSE_ID' in df_clean.columns:
    df_clean['license_matched'] = df_clean['BL_LICENSE_ID'].notna()
elif 'LICENSE NUMBER' in df_clean.columns:
    df_clean['license_matched'] = df_clean['LICENSE NUMBER'].notna()
else:
    df_clean['license_matched'] = False

if 'Inspection Date' in df_clean.columns:
    prior_key = None
    for candidate in ['BL_LICENSE_ID', 'License #', 'LICENSE NUMBER', 'DBA Name', 'AKA Name']:
        if candidate in df_clean.columns:
            prior_key = candidate
            break

    if prior_key is not None:
        ordered_idx = df_clean.sort_values([prior_key, 'Inspection Date']).index
        has_prior = pd.Series(False, index=df_clean.index)
        has_prior.loc[ordered_idx] = (
            df_clean.loc[ordered_idx]
            .groupby(prior_key)
            .cumcount()
            .gt(0)
            .to_numpy()
        )
        df_clean['has_prior_inspection'] = has_prior
    else:
        df_clean['has_prior_inspection'] = False

if 'Location' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Location'])

log_step(
    'cast_types_and_feature_safety',
    rows_before,
    df_clean.shape[0],
    cols_before,
    df_clean.shape[1],
    note='Type casting, Risk standardization, leakage fix, informative flags, drop Location'
)
print('Shape after type and value normalization:', df_clean.shape)
print('Rows with nulled future LICENSE TERM START DATE:', int(future_license_start_mask.sum()))

Shape after type and value normalization: (196676, 57)
Rows with nulled future LICENSE TERM START DATE: 145793


In [ ]:
# Action 6: Quarantine non-trainable labels and final validation snapshot
rows_before, cols_before = df_clean.shape

if 'Results' in df_clean.columns:
    results_clean = df_clean['Results'].astype('string').str.strip()
    quarantine_mask = results_clean.isna() | results_clean.eq('') | results_clean.isin(['Not Ready', 'Business Not Located'])
    quarantine_df = df_clean.loc[quarantine_mask].copy()

    if not quarantine_df.empty:
        quarantine_df['quarantine_reason'] = 'missing_or_non_trainable_result'

    df_clean = df_clean.loc[~quarantine_mask].copy()
    df_clean['Results'] = results_clean.loc[~quarantine_mask]
else:
    quarantine_df = pd.DataFrame()

log_step(
    'quarantine_non_trainable_labels',
    rows_before,
    df_clean.shape[0],
    cols_before,
    df_clean.shape[1],
    note='Removed missing / Not Ready / Business Not Located labels from training set'
)

post_profile = profile(df_clean, 'Post-clean profile')

print('\nPost-clean checks:')
print('Quarantined rows:', len(quarantine_df))
print('Risk missing:', int(df_clean['Risk'].isna().sum()) if 'Risk' in df_clean.columns else 'N/A')
print('Risk levels:', df_clean['Risk'].value_counts(dropna=False).to_dict() if 'Risk' in df_clean.columns else 'N/A')
print('Malformed Zip count:', int(df_clean['Zip'].dropna().str.match(r'^\d{5}$').eq(False).sum()) if 'Zip' in df_clean.columns else 'N/A')
print('Future LICENSE TERM START DATE count:', int(((df_clean['Inspection Date'].notna()) & (df_clean['LICENSE TERM START DATE'].notna()) & (df_clean['LICENSE TERM START DATE'] > df_clean['Inspection Date'])).sum()) if {'Inspection Date', 'LICENSE TERM START DATE'}.issubset(df_clean.columns) else 'N/A')
print('Violations recorded flag present:', 'violations_recorded' in df_clean.columns)
print('Has prior inspection flag present:', 'has_prior_inspection' in df_clean.columns)
print('License matched flag present:', 'license_matched' in df_clean.columns)
print('Longitude flagged:', int(df_clean['flag_longitude_outside_typical_range'].sum()) if 'flag_longitude_outside_typical_range' in df_clean.columns else 'N/A')
print('Duplicate Inspection ID:', int(df_clean.duplicated(subset=['Inspection ID']).sum()))

post_profile.head(20)

=== Post-clean profile ===
shape: (194701, 57)
full duplicates: 0
duplicate Inspection ID: 0

Post-clean checks:
Quarantined rows: 1975
Risk missing: 0
Risk levels: {'High': 139963, 'Medium': 37870, 'Low': 16816, 'Unknown': 52}
Malformed Zip count: 0
Future LICENSE TERM START DATE count: 0
Violations recorded flag present: True
Has prior inspection flag present: True
License matched flag present: True
Longitude flagged: 0
Duplicate Inspection ID: 0


,missing_count,missing_pct,dtype
APPLICATION CREATED DATE,176601,90.70,datetime64[ns]
LICENSE STATUS CHANGE DATE,154180,79.19,datetime64[ns]
LICENSE TERM START DATE,153665,78.92,datetime64[ns]
SSA,133477,68.55,Int64
Violations,50310,25.84,object
BUSINESS ACTIVITY,20414,10.48,object
BUSINESS ACTIVITY ID,20414,10.48,object
NEIGHBORHOOD,11596,5.96,object
COMMUNITY AREA,11369,5.84,Int64
COMMUNITY AREA NAME,11369,5.84,object


In [ ]:
# Save cleaned data, quarantine data, and cleaning log
if 'PROCESSED_PARQUET_PATH' not in globals():
    PROCESSED_PARQUET_PATH = Path('../data/processed/merged_inspections_licenses_inner_clean.parquet')
if 'QUARANTINE_PATH' not in globals():
    QUARANTINE_PATH = Path('../data/interim/quarantine.csv')

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
QUARANTINE_PATH.parent.mkdir(parents=True, exist_ok=True)

cleaning_log = pd.DataFrame(log_rows)
df_clean.to_csv(PROCESSED_PATH, index=False)
df_clean.to_parquet(PROCESSED_PARQUET_PATH, index=False)
quarantine_df.to_csv(QUARANTINE_PATH, index=False)
cleaning_log.to_csv(LOG_PATH, index=False)

print('Saved cleaned data (csv) to:', PROCESSED_PATH)
print('Saved cleaned data (parquet) to:', PROCESSED_PARQUET_PATH)
print('Saved quarantine rows to:', QUARANTINE_PATH)
print('Saved cleaning log to:', LOG_PATH)
cleaning_log

Saved cleaned data (csv) to: ..\data\processed\merged_inspections_licenses_inner_clean.csv
Saved cleaned data (parquet) to: ..\data\processed\merged_inspections_licenses_inner_clean.parquet
Saved quarantine rows to: ..\data\interim\quarantine.csv
Saved cleaning log to: ..\data\interim\cleaning_log.csv


,step,rows_before,rows_after,rows_removed,cols_before,cols_after,cols_removed,note
0,drop_all_null_columns,196825,196825,0,59,54,5,Historical Wards 2003-2015; Zip Codes; Communi...
1,drop_exact_duplicates,196825,196676,149,54,54,0,Drop fully duplicated rows
2,drop_duplicate_inspection_id,196676,196676,0,54,54,0,Keep latest Inspection Date per Inspection ID
3,add_longitude_flag,196676,196676,0,54,55,-1,Flag out-of-range longitude values
4,cast_types_zip_risk_location,196676,196676,0,55,55,0,Type casting + Zip + Risk + Location
5,drop_exact_duplicates,196676,196676,0,55,55,0,Drop fully duplicated rows
6,drop_duplicate_inspection_id,196676,196676,0,55,55,0,Keep latest Inspection Date per Inspection ID
7,normalize_text_fields_and_flags,196676,196676,0,55,55,0,"Normalize text, add flags, drop City/State"
8,cast_types_and_feature_safety,196676,196676,0,55,57,-2,"Type casting, Risk standardization, leakage fi..."
9,quarantine_non_trainable_labels,196676,194701,1975,57,57,0,Removed missing / Not Ready / Business Not Loc...
